In [1]:
import pandas as pd

data_frame = pd.read_csv('output/1.10_preprocessed_train.csv')


In [2]:
import numpy as np

df = data_frame.copy()
df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age__was_missing,Cabin__was_missing,Embarked__was_missing,...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,1,0,3,22.0,1.0,0,7.2500,0,1,0,...,False,False,False,False,False,False,False,False,False,True
1,2,1,1,38.0,1.0,0,65.6344,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,3,1,3,26.0,0.0,0,7.9250,0,1,0,...,False,False,False,False,False,False,False,False,False,True
3,4,1,1,35.0,1.0,0,53.1000,0,0,0,...,False,False,False,False,False,False,False,False,False,True
4,5,0,3,35.0,0.0,0,8.0500,0,1,0,...,False,False,False,False,False,False,False,False,False,True


# 1.11 Data Mining (Model Building)

We will follow the course steps:

1. Select an appropriate modeling technique (classification / regression / clustering)
2. Build (train) a model
3. Evaluate on unseen data
4. Tune hyperparameters (simple GridSearch)
5. Iterate if needed

## 3.1 Select the Modeling Technique
We first choose a *task* based on whether we have a target column:
- If we find a target (e.g., `Survived`), we do **supervised learning** (classification/regression).
- If there is no target, we show a **clustering** demo (unsupervised).

In [3]:
# Try to detect a target column automatically (you can change this manually)
target_candidates = ['Survived', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df.columns), None)

target

'Survived'

In [4]:
# If target exists: decide if it's classification or regression
task = None
if target is None:
    task = 'clustering'
else:
    y = df[target]
    # Heuristic: few unique values => classification, otherwise regression
    n_unique = y.nunique(dropna=True)
    if n_unique <= 20:
        task = 'classification'
    else:
        task = 'regression'

task

'classification'

## 3.2 Build the Model (Training)

We split the data into train/test and fit a simple baseline model.

In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

if task in ['classification', 'regression']:
    # X = all columns except target
    X = df.drop(columns=[target]).copy()
    y = df[target].copy()

    # Ensure X is numeric (preprocessed data should already be numeric)
    X = X.select_dtypes(include=[np.number]).copy()

    stratify = y if task == 'classification' else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
    )
    X_train.shape, X_test.shape
else:
    print('No supervised target found: we will do clustering later.')

In [6]:
if task == 'classification':
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier

    # Baseline models
    lr = LogisticRegression(max_iter=2000)
    rf = RandomForestClassifier(random_state=RANDOM_STATE)

    lr.fit(X_train, y_train)
    rf.fit(X_train, y_train)

    print('Models trained: LogisticRegression, RandomForestClassifier')
elif task == 'regression':
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor

    lr = LinearRegression()
    rf = RandomForestRegressor(random_state=RANDOM_STATE)

    lr.fit(X_train, y_train)
    rf.fit(X_train, y_train)

    print('Models trained: LinearRegression, RandomForestRegressor')

Models trained: LogisticRegression, RandomForestClassifier


## 3.4 Evaluate on Unseen Data
We evaluate baseline models on the test set using common metrics.

In [7]:
if task == 'classification':
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    for name, model in [('LogisticRegression', lr), ('RandomForest', rf)]:
        y_pred = model.predict(X_test)
        print('===', name, '===')
        print('Accuracy:', accuracy_score(y_test, y_pred))
        print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))
elif task == 'regression':
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    for name, model in [('LinearRegression', lr), ('RandomForest', rf)]:
        y_pred = model.predict(X_test)
        print('===', name, '===')
        print('MAE:', mean_absolute_error(y_test, y_pred))
        print('RMSE:', mean_squared_error(y_test, y_pred, squared=False))
        print('R2:', r2_score(y_test, y_pred))

=== LogisticRegression ===
Accuracy: 0.6703910614525139
Confusion matrix:
 [[88 22]
 [37 32]]
              precision    recall  f1-score   support

           0       0.70      0.80      0.75       110
           1       0.59      0.46      0.52        69

    accuracy                           0.67       179
   macro avg       0.65      0.63      0.63       179
weighted avg       0.66      0.67      0.66       179

=== RandomForest ===
Accuracy: 0.6312849162011173
Confusion matrix:
 [[79 31]
 [35 34]]
              precision    recall  f1-score   support

           0       0.69      0.72      0.71       110
           1       0.52      0.49      0.51        69

    accuracy                           0.63       179
   macro avg       0.61      0.61      0.61       179
weighted avg       0.63      0.63      0.63       179



## 3.3 Hyperparameter Tuning (Simple Grid Search)

We tune the Random Forest with a small grid to keep it simple.

In [8]:
from sklearn.model_selection import GridSearchCV

if task == 'classification':
    from sklearn.ensemble import RandomForestClassifier
    base = RandomForestClassifier(random_state=RANDOM_STATE)
    param_grid = {
        'n_estimators': [100, 300],
        'max_depth': [None, 5, 10],
    }
    grid = GridSearchCV(base, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    print('Best params:', grid.best_params_)
elif task == 'regression':
    from sklearn.ensemble import RandomForestRegressor
    base = RandomForestRegressor(random_state=RANDOM_STATE)
    param_grid = {
        'n_estimators': [100, 300],
        'max_depth': [None, 5, 10],
    }
    grid = GridSearchCV(base, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    print('Best params:', grid.best_params_)

Best params: {'max_depth': 5, 'n_estimators': 300}


In [9]:
if task == 'classification':
    from sklearn.metrics import accuracy_score
    y_pred = best_model.predict(X_test)
    print('Tuned RandomForest Accuracy:', accuracy_score(y_test, y_pred))
elif task == 'regression':
    from sklearn.metrics import mean_absolute_error
    y_pred = best_model.predict(X_test)
    print('Tuned RandomForest MAE:', mean_absolute_error(y_test, y_pred))

Tuned RandomForest Accuracy: 0.6815642458100558


## 3.5 Iterate (Simple Error Analysis)

We quickly inspect where the model makes mistakes (classification) or large errors (regression).

In [10]:
if task == 'classification':
    y_pred = best_model.predict(X_test)
    wrong_idx = X_test.index[y_pred != y_test]
    print('Number of mistakes:', len(wrong_idx))
    # Show a few rows where the model is wrong
    df.loc[wrong_idx].head(10)
elif task == 'regression':
    y_pred = best_model.predict(X_test)
    errors = (y_test - y_pred).abs()
    worst = errors.sort_values(ascending=False).head(10).index
    print('Worst errors (top 10):')
    df.loc[worst].head(10)
else:
    print('Skipping supervised error analysis.')

Number of mistakes: 57


## Unsupervised Demo (If No Target): Clustering

If no target column exists, we can still mine patterns using clustering (KMeans) and evaluate with silhouette score.

In [11]:
if task == 'clustering':
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    Xc = df.select_dtypes(include=[np.number]).copy()
    Xc = Xc.dropna()

    kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init='auto')
    labels = kmeans.fit_predict(Xc)
    print('Silhouette score:', silhouette_score(Xc, labels))
    pd.Series(labels).value_counts()
else:
    print('Target found, so we focus on supervised learning.')

Target found, so we focus on supervised learning.


## (Optional) Save the Best Model

Saving helps reuse the trained model later (deployment, API, etc.).

In [12]:
# Optional: save the tuned model
import joblib

if task in ['classification', 'regression']:
    model_path = 'output/1.11_best_model.joblib'
    joblib.dump(best_model, model_path)
    model_path
else:
    print('No supervised model to save.')